# Feature Engineering

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

In [3]:
# Create target variable (shifted -1h)
df["target_pm25_1h"] = df.groupby("station")["PM2.5"].shift(-1)

In [4]:
# Create lag for target and momentum features
lag_target = [1, 2, 24]

for k in lag_target:
    df[f"pm25_lag_{k}h"] = df.groupby("station")["PM2.5"].shift(k)

df["pm25_diff_1h"] = df["PM2.5"] - df["pm25_lag_1h"]

In [5]:
# Create mean rolling window for target column
for k in [6, 24]:
    df[f"pm25_roll_mean_{k}h"] = df.groupby("station")["PM2.5"].transform(lambda x: x.rolling(k).mean())

In [6]:
# Encode hour and month columns into cyclinical time
def sin(time, period):
    return np.sin((2 * np.pi * time) / period)

def cos(time, period):
    return np.cos((2 * np.pi * time) / period)

df["hour_sin"] = sin(df["hour"], 24)
df["hour_cos"] = cos(df["hour"], 24)

df["month_sin"] = sin(df["month"], 12)
df["month_cos"] = cos(df["month"], 12)

df["humid"] = df["TEMP"] - df["DEWP"]

In [7]:
# Drop the remaining NaN values
df_fe = df.dropna().reset_index(drop=True)

print(f"Before: {df.shape}")
print(f"After: {df_fe.shape}")

Before: (105192, 30)
After: (100778, 30)


In [8]:
# Encode categorical data
df_fe = pd.get_dummies(df_fe, columns=['station', 'wd'], dtype=int)

In [9]:
# Save featured engineering dataset
#df_fe.to_csv("../data/processed/fe_data.csv", index=False)